In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, LongType


schema = StructType([
    StructField('_c0', IntegerType(), True),            # índice
    StructField('trans_date_trans_time', StringType(), True), # fecha y hora
    StructField('cc_num', LongType(), True),            # número de tarjeta
    StructField('merchant', StringType(), True),        # nombre del comercio
    StructField('category', StringType(), True),        # categoría
    StructField('amt', DoubleType(), True),             # monto
    StructField('first', StringType(), True),           # nombre titular
    StructField('last', StringType(), True),            # apellido titular
    StructField('gender', StringType(), True),          # género
    StructField('street', StringType(), True),          # calle
    StructField('city', StringType(), True),            # ciudad
    StructField('state', StringType(), True),           # estado
    StructField('zip', IntegerType(), True),            # código postal
    StructField('lat', DoubleType(), True),             # latitud titular
    StructField('long', DoubleType(), True),            # longitud titular
    StructField('city_pop', IntegerType(), True),       # población ciudad
    StructField('job', StringType(), True),             # ocupación
    StructField('dob', StringType(), True),             # fecha nacimiento
    StructField('trans_num', StringType(), True),       # código transacción
    StructField('unix_time', LongType(), True),         # tiempo unix
    StructField('merch_lat', DoubleType(), True),       # latitud comercio
    StructField('merch_long', DoubleType(), True),      # longitud comercio
    StructField('is_fraud', IntegerType(), True)        # 0=legítima, 1=fraude
])


print('Esquema definido correctamente')
df_temp = spark.createDataFrame([], schema)
df_temp.printSchema()


In [0]:
try:
    df = spark.read.format('csv') \
        .option('header', 'true') \
        .schema(schema) \
        .load('/Volumes/fraud_project/bronze/raw_data/input/csv/fraudTest.csv')


    print('Archivo leído correctamente')
    print(f'Total de registros: {df.count()}')


except Exception as e:
    print(f'Error al leer el archivo: {str(e)}')


In [0]:
from pyspark.sql.functions import col


try:
    # Registros válidos: is_fraud debe ser 0 o 1
    df_validos = df.filter(col('is_fraud').isin([0, 1]))


    # Registros inválidos: faltan datos esenciales para analizar el fraude
    df_invalidos = df.filter(
        col('amt').isNull() |
        col('is_fraud').isNull() |
        col('trans_date_trans_time').isNull() |
        col('merchant').isNull()
    )


    print(f'Registros válidos: {df_validos.count()}')
    print(f'Registros descartados: {df_invalidos.count()}')


except Exception as e:
    print(f'Error al separar los datos: {str(e)}')


In [0]:
try:
    df_validos.write.format('delta').mode('overwrite') \
        .saveAsTable('fraud_project.bronze.transacciones_validas')


    df_invalidos.write.format('delta').mode('overwrite') \
        .saveAsTable('fraud_project.bronze.transacciones_con_errores')


    print('Tablas guardadas correctamente')
    print(f'Registros válidos guardados: {df_validos.count()}')
    print(f'Registros con errores: {df_invalidos.count()}')


except Exception as e:
    print(f'Error al guardar las tablas: {str(e)}')


In [0]:
try:
    print('=== Vista de los datos válidos ===')
    display(df_validos)


    print('\n=== Conteo por tipo de fraude ===')
    df_validos.groupBy('is_fraud').count().show()


    print('\n=== Columnas del dataset ===')
    print(df_validos.columns)


except Exception as e:
    print(f'Error al visualizar: {str(e)}')
